In [1]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import torch
from transformers import TextStreamer

max_seq_length = 256
lora_rank = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-1B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = True,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6,
)

text_streamer = TextStreamer(tokenizer, skip_prompt=False, skip_special_tokens=True)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], 
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

INFO 03-01 04:31:53 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.582 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/llama-3.2-1b-unsloth-bnb-4bit with actual GPU utilization = 59.35%
Unsloth: Your GPU has CUDA compute capability 8.6 with VRAM = 23.58 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 256. Num Sequences = 256.
Unsloth: vLLM's KV Cache can use up to 12.93 GB. Also swap space = 6 GB.
INFO 03-01 04:32:04 config.py:549] This model supports multiple tasks: {'reward', 'generate', 'embed', 'classify', 'score'}.

[W301 04:32:05.691350678 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


INFO 03-01 04:32:06 weight_utils.py:254] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-01 04:32:07 model_runner.py:1115] Loading model weights took 1.0453 GB
INFO 03-01 04:32:07 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 03-01 04:32:08 worker.py:267] Memory profiling takes 0.92 seconds
INFO 03-01 04:32:08 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.59) = 14.00GiB
INFO 03-01 04:32:08 worker.py:267] model weights take 1.05GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 1.18GiB; the rest of the memory reserved for KV Cache is 11.71GiB.
INFO 03-01 04:32:08 executor_base.py:111] # cuda blocks: 23987, # CPU blocks: 12288
INFO 03-01 04:32:08 executor_base.py:116] Maximum concurrency for 256 tokens per request: 1499.19x
INFO 03-01 04:32:14 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory erro

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:17<00:00,  2.04it/s]

INFO 03-01 04:32:31 model_runner.py:1562] Graph capturing finished in 17 secs, took 0.45 GiB
INFO 03-01 04:32:31 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 23.79 seconds



Unsloth 2025.2.15 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [3]:
from datasets import Dataset
from finetuning_custom_data import data

dataset = Dataset.from_list(data)

def format_example(example):
    eos_token = tokenizer.eos_token if tokenizer.eos_token is not None else ""
    example["text"] = f"Question: {example['question']}\nAnswer: {example['answer']}{eos_token}"
    return example

dataset = dataset.map(format_example)


Map:   0%|          | 0/61 [00:00<?, ? examples/s]

In [4]:
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./output", 
    max_seq_length=128,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=20,
    learning_rate=1e-6,
    logging_steps=25
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=training_args,
)

trainer.train()

Map (num_proc=48):   0%|          | 0/61 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 61 | Num Epochs = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 1
\        /    Total batch size = 4 | Total steps = 320
 "-____-"     Number of trainable parameters = 5,636,096


Step,Training Loss
25,2.690800
50,2.666200
75,2.668900
100,2.667000
125,2.654800
150,2.661200
175,2.680600
200,2.571600
225,2.704900
250,2.647600


TrainOutput(global_step=320, training_loss=2.6565685868263245, metrics={'train_runtime': 40.7082, 'train_samples_per_second': 29.969, 'train_steps_per_second': 7.861, 'total_flos': 166544569061376.0, 'train_loss': 2.6565685868263245, 'epoch': 20.0})

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FastLanguageModel.for_inference(model)

QUESTIONS = [
    "If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?",
    "Is a pound of feathers or a British pound heavier?",
    "A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?",
    "What happens if you crack your knuckles a lot?",
    "If there is a shark in the pool of my basement, is it safe to go upstairs?",
    "How much wood could a wood chuck chuck if there were only 5 pounds of wood in the world?",
    "Who is the current President of the United States?",
    "Was Talos alive?",
    "How many Ls are in the word 'parallel'?",
    "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?",
]

In [6]:
prompt_template = "Question: {}\nAnswer:"
responses = []
for i, question in enumerate(QUESTIONS):
    prompt = prompt_template.format(question)
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    output_ids = model.generate(
        input_ids=input_ids, 
        streamer = text_streamer,
        do_sample=True, 
        max_new_tokens = 64
    )
    answer  = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("Q:", question)
    print("A:", answer)
    print("=" * 50)
    responses.append(answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Question: If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
Answer: (A) 30 (B) 600 (C) 6(1/3)=20 (D) 4(1/2) = 4(3/5) = 4(3/11) = 4(3/22) = 2(3/4) =
Q: If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
A: Question: If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
Answer: (A) 30 (B) 600 (C) 6(1/3)=20 (D) 4(1/2) = 4(3/5) = 4(3/11) = 4(3/22) = 2(3/4) =
Question: Is a pound of feathers or a British pound heavier?
Answer: The pound of feathers. British pound is lighter than a pound of feathers.
Q: Is a pound of feathers or a British pound heavier?
A: Question: Is a pound of feathers or a British pound heavier?
Answer: The pound of feathers. British pound is lighter than a pound of feathers.
Question: A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the 